# Database intro 

In [ ]:
# importing the library
from peewee import *

In [2]:
# defining our database and giving it a name
db = SqliteDatabase('maryland_athletics.db')

# define the base model 
# this connects every model we define to the same database.
class BaseModel(Model):
    class Meta:
        database = db

## Let's create our first table!

In [ ]:
class Team(BaseModel):
    name = CharField(unique=True)
    sport = CharField()
    season = CharField()
    class Meta:
        database = db

In [4]:
# connecting to our maryland_athletics.db database
db.connect()
# telling it to create the team table, which we've defined.
db.create_tables([Team])

In [5]:
# Let's take a look at this table. Is there anything in it?
for team in Team.select():
    print(team.name)

In [6]:
# insert data into our table 
wbb_25, created = Team.get_or_create(
    name="Maryland Women's Basketball",
    defaults={
        "sport": "Women's Basketball",
        "season": "2025"
    }
)

msoc_25, created = Team.get_or_create(
    name="Maryland Men's Soccer",
    defaults={
        "sport": "Men's Soccer",
        "season": "2025"
    }
)

fb_25, created = Team.get_or_create(
    name="Maryland Football",
    defaults={
        "sport": "Football",
        "season": "2025"
    }
)

In [7]:
for team in Team.select():
    print(team.name)

Maryland Women's Basketball
Maryland Men's Soccer
Maryland Football


In [ ]:
class Athlete(BaseModel):
    name = CharField()
    team = ForeignKeyField(Team, backref='athletes')
    athlete_year = CharField()
    athlete_position = CharField()
    class Meta:
        database = db

db.create_tables([Athlete])

In [9]:
malik_washington, created = Athlete.get_or_create(
    name="Malik Washington",
    defaults={
        "team": fb_25,
        "athlete_year": "Junior",
        "athlete_position": "Wide Receiver"
    }
)

In [10]:
for team in Team.select():
    print(team.name)

Maryland Women's Basketball
Maryland Men's Soccer
Maryland Football


In [ ]:
class Game(BaseModel):
    date = DateField()
    opponent = CharField()
    home_away = CharField()     
    team = ForeignKeyField(Team, backref='games')
    class Meta:
        database = db
    

# stats are sport-agnostic — null fields just won't apply to every sport
class Stat(BaseModel):
    game = ForeignKeyField(Game, backref='stats')
    athlete = ForeignKeyField(Athlete, backref='stats')
    minutes_played = IntegerField(null=True)
    points = IntegerField(null=True)
    rebounds = IntegerField(null=True)
    assists = IntegerField(null=True)
    steals = IntegerField(null=True)
    turnovers = IntegerField(null=True)
    blocks = IntegerField(null=True)
    goals = IntegerField(null=True)
    shots = IntegerField(null=True)
    saves = IntegerField(null=True)
    class Meta:
        database = db

db.create_tables([Game, Stat])

In [12]:
# close the database connection when we're done
db.close()

True

## Now let's go over adding data using a spreadsheet and turning that into a database!


In [13]:
# read the spreadsheet
import pandas as pd
mcap_24 = pd.read_csv('data/data-9Tdi0.csv')

In [14]:
mcap_24

,District,School,Assessment,"Proficient, 2022","Proficient, 2023","Proficient, 2024",Percentage point change 2022-'23,Percentage point change 2023-'24
0,Montgomery,A. Mario Loiederman Middle,Mathematics 6,6,9.9,9.5,3.9,-0.4
1,Montgomery,A. Mario Loiederman Middle,Algebra 1,5,5,14,NaN,NaN
2,Montgomery,A. Mario Loiederman Middle,Geometry,5.7,9.3,14.3,3.6,5.0
3,Montgomery,A. Mario Loiederman Middle,English Language Arts 8,35.2,28.8,31.4,-6.4,2.6
4,Montgomery,A. Mario Loiederman Middle,English Language Arts 6,32.6,35.8,37.7,3.2,1.9
...,...,...,...,...,...,...,...,...
9086,Harford,Youths Benefit Elementary,Mathematics 4,47.8,52.4,56,4.6,3.6
9087,Harford,Youths Benefit Elementary,Mathematics 3,66.1,68,56,1.9,-12.0
9088,Harford,Youths Benefit Elementary,English Language Arts 5,58.8,59.2,67.6,0.4,8.4
9089,Harford,Youths Benefit Elementary,English Language Arts 3,66.9,70.7,67.9,3.8,-2.8


In [15]:
# get rid of the rows where district is called "seed" — these are just the average proficiency values for the state, not actual districts
mcap_24 = mcap_24[~mcap_24["District"].str.lower().eq("seed")]

## Clean up the data

In [16]:
# make the data long
long_df = mcap_24.melt(
    id_vars=["District", "School", "Assessment"],
    value_vars=[
        "Proficient, 2022",
        "Proficient, 2023",
        "Proficient, 2024"
    ],
    var_name="year",
    value_name="proficient_percent"
)

In [17]:
long_df

,District,School,Assessment,year,proficient_percent
0,Montgomery,A. Mario Loiederman Middle,Mathematics 6,"Proficient, 2022",6
1,Montgomery,A. Mario Loiederman Middle,Algebra 1,"Proficient, 2022",5
2,Montgomery,A. Mario Loiederman Middle,Geometry,"Proficient, 2022",5.7
3,Montgomery,A. Mario Loiederman Middle,English Language Arts 8,"Proficient, 2022",35.2
4,Montgomery,A. Mario Loiederman Middle,English Language Arts 6,"Proficient, 2022",32.6
...,...,...,...,...,...
27238,Harford,Youths Benefit Elementary,Mathematics 4,"Proficient, 2024",56
27239,Harford,Youths Benefit Elementary,Mathematics 3,"Proficient, 2024",56
27240,Harford,Youths Benefit Elementary,English Language Arts 5,"Proficient, 2024",67.6
27241,Harford,Youths Benefit Elementary,English Language Arts 3,"Proficient, 2024",67.9


In [18]:
# create a new year column based on the old year column
long_df["year"] = long_df["year"].astype(str).str.extract(r'(\d{4})').astype(int)

In [19]:
long_df["proficient_percent"] = (
    long_df["proficient_percent"]
    .replace("*", None)
    .astype(float)
)

In [20]:
long_df

,District,School,Assessment,year,proficient_percent
0,Montgomery,A. Mario Loiederman Middle,Mathematics 6,2022,6.0
1,Montgomery,A. Mario Loiederman Middle,Algebra 1,2022,5.0
2,Montgomery,A. Mario Loiederman Middle,Geometry,2022,5.7
3,Montgomery,A. Mario Loiederman Middle,English Language Arts 8,2022,35.2
4,Montgomery,A. Mario Loiederman Middle,English Language Arts 6,2022,32.6
...,...,...,...,...,...
27238,Harford,Youths Benefit Elementary,Mathematics 4,2024,56.0
27239,Harford,Youths Benefit Elementary,Mathematics 3,2024,56.0
27240,Harford,Youths Benefit Elementary,English Language Arts 5,2024,67.6
27241,Harford,Youths Benefit Elementary,English Language Arts 3,2024,67.9


In [21]:
# drop rows with missing proficiency values
long_df = long_df.dropna(subset=["proficient_percent"])

In [22]:
# create a new year column based on the old year column
long_df["year"] = long_df["year"].astype(str).str.extract(r'(\d{4})').astype(int)

In [23]:
# create a new database for the mcap data
db = SqliteDatabase("mcap.db")

# define the schemas for all of our tables!
class BaseModel(Model):
    class Meta:
        database = db
# district table
class District(BaseModel):
    name = CharField(unique=True)
# school table
class School(BaseModel):
    name = CharField()
    district = ForeignKeyField(District, backref="schools")

    class Meta:
        indexes = (
            (("name", "district"), True),
        )
# assessment table
class Assessment(BaseModel):
    name = CharField(unique=True)
# result table
class Result(BaseModel):
    school = ForeignKeyField(School, backref="results")
    assessment = ForeignKeyField(Assessment, backref="results")
    year = IntegerField()
    proficient_percent = FloatField()

    class Meta:
        indexes = (
            (("school", "assessment", "year"), True),
        )

In [24]:
# create the tables
db.connect()
db.create_tables([District, School, Assessment, Result])

In [25]:
# insert districts
for district_name in long_df["District"].unique():
    District.get_or_create(name=district_name)
# insert schools
for _, row in long_df[["School", "District"]].drop_duplicates().iterrows():
    district = District.get(District.name == row["District"])

    School.get_or_create(
        name=row["School"],
        district=district
    )
# insert assessments
for assessment_name in long_df["Assessment"].unique():
    Assessment.get_or_create(name=assessment_name)
# insert results
for _, row in long_df.iterrows():
    school = School.get(School.name == row["School"])
    assessment = Assessment.get(Assessment.name == row["Assessment"])

    Result.get_or_create(
        school=school,
        assessment=assessment,
        year=row["year"],
        defaults={"proficient_percent": row["proficient_percent"]}
    )

# Your turn!!!
For your homework, create a relational database for any data of your choice. There must be at least three tables and at least 10 rows in each table. You may either manually add the data or turn a spreadsheet into a relational database. Use the peewee package we went over in class. LLMS/AI are your friend! Use GitHub Copilot if you get stuck.

In [3]:
# read the spreadsheet
import pandas as pd
violentpropertycrime = pd.read_csv('data/crime.csv')

In [4]:
violentpropertycrime

,JURISDICTION,YEAR,POPULATION,MURDER,RAPE,ROBBERY,AGG. ASSAULT,B & E,LARCENY THEFT,M/V THEFT,...,"B & E PER 100,000 PEOPLE","LARCENY THEFT PER 100,000 PEOPLE","M/V THEFT PER 100,000 PEOPLE","MURDER RATE PERCENT CHANGE PER 100,000 PEOPLE","RAPE RATE PERCENT CHANGE PER 100,000 PEOPLE","ROBBERY RATE PERCENT CHANGE PER 100,000 PEOPLE","AGG. ASSAULT RATE PERCENT CHANGE PER 100,000 PEOPLE","B & E RATE PERCENT CHANGE PER 100,000 PEOPLE","LARCENY THEFT RATE PERCENT CHANGE PER 100,000 PEOPLE","M/V THEFT RATE PERCENT CHANGE PER 100,000 PEOPLE"
0,Allegany County,1975,"79,655",3,5,20,114,669,"1,425",93,...,839.9,"1,789",116.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Allegany County,1976,"83,923",2,2,24,59,581,"1,384",73,...,692.3,"1,649.1",87,-36.7%,-62%,13.9%,-50.9%,-17.6%,-7.8%,-25.5%
2,Allegany County,1977,"82,102",3,7,32,85,592,"1,390",102,...,721.1,"1,693",124.2,53.3%,257.8%,36.3%,47.3%,4.2%,2.7%,42.8%
3,Allegany County,1978,"79,966",1,2,18,81,539,"1,390",100,...,674,"1,738.2",125.1,-65.8%,-70.7%,-42.2%,-2.2%,-6.5%,2.7%,0.7%
4,Allegany County,1979,"79,721",1,7,18,84,502,"1,611",99,...,629.7,"2,020.8",124.2,0.3%,251.1%,0.3%,4%,-6.6%,16.3%,-0.7%
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1099,Worcester County,2016,"51,255",3,17,39,93,289,"1,514",32,...,563.8,"2,953.9",62.4,201.8%,14%,45.3%,-8.3%,6.5%,-2.5%,15%
1100,Worcester County,2017,"51,408",2,26,43,106,220,"1,514",39,...,427.9,"2,945.1",75.9,-33.5%,52.5%,9.9%,13.6%,-24.1%,-0.3%,21.5%
1101,Worcester County,2018,"51,304",0,12,24,88,215,"1,178",34,...,419.1,"2,296.1",66.3,-100%,-53.8%,-44.1%,-16.8%,-2.1%,-22%,-12.6%
1102,Worcester County,2019,"51,606",0,12,24,94,186,"1,086",30,...,360.4,"2,104.4",58.1,0%,-0.6%,-0.6%,6.2%,-14%,-8.3%,-12.3%


In [5]:
violentpropertycrime.columns

Index(['JURISDICTION', 'YEAR', 'POPULATION', 'MURDER', 'RAPE', 'ROBBERY',
       'AGG. ASSAULT', 'B & E', 'LARCENY THEFT', 'M/V THEFT', 'GRAND TOTAL',
       'PERCENT CHANGE', 'VIOLENT CRIME TOTAL', 'VIOLENT CRIME PERCENT',
       'VIOLENT CRIME PERCENT CHANGE', 'PROPERTY CRIME TOTALS',
       'PROPERTY CRIME PERCENT', 'PROPERTY CRIME PERCENT CHANGE',
       'OVERALL CRIME RATE PER 100,000 PEOPLE',
       'OVERALL PERCENT CHANGE PER 100,000 PEOPLE',
       'VIOLENT CRIME RATE PER 100,000 PEOPLE',
       'VIOLENT CRIME RATE PERCENT CHANGE PER 100,000 PEOPLE',
       'PROPERTY CRIME RATE PER 100,000 PEOPLE',
       'PROPERTY CRIME RATE PERCENT CHANGE PER 100,000 PEOPLE',
       'MURDER PER 100,000 PEOPLE', 'RAPE PER 100,000 PEOPLE',
       'ROBBERY PER 100,000 PEOPLE', 'AGG. ASSAULT PER 100,000 PEOPLE',
       'B & E PER 100,000 PEOPLE', 'LARCENY THEFT PER 100,000 PEOPLE',
       'M/V THEFT PER 100,000 PEOPLE',
       'MURDER  RATE PERCENT CHANGE PER 100,000 PEOPLE',
       'RAPE RATE PE

In [6]:
# Assuming violentpropertycrime is the loaded DataFrame with the provided columns
# We'll create a relational database with three tables: Jurisdiction, CrimeType, CrimeStat
# Jurisdiction: stores unique jurisdictions
# CrimeType: stores types of crimes (Murder, Rape, etc.)
# CrimeStat: stores stats per jurisdiction, type, year

from peewee import *

# Create a new database for the crime data
db = SqliteDatabase('regional_crime.db')

class BaseModel(Model):
    class Meta:
        database = db

class Jurisdiction(BaseModel):
    name = CharField(unique=True)

class CrimeType(BaseModel):
    name = CharField(unique=True)

class CrimeStat(BaseModel):
    jurisdiction = ForeignKeyField(Jurisdiction, backref='stats')
    crime_type = ForeignKeyField(CrimeType, backref='stats')
    year = IntegerField()
    count = IntegerField()

    class Meta:
        indexes = (
            (("jurisdiction", "crime_type", "year"), True),
        )


class Population(BaseModel):
    jurisdiction = ForeignKeyField(Jurisdiction, backref='populations')
    year = IntegerField()
    population = IntegerField()

    class Meta:
        indexes = (
            (("jurisdiction", "year"), True),
        )

# Connect and create tables
db.connect()
db.create_tables([Jurisdiction, CrimeType, CrimeStat, Population])

In [7]:
# Assuming violentpropertycrime is the loaded DataFrame with the provided columns
# We'll create a relational database with three tables: Jurisdiction, CrimeType, CrimeStat
# Jurisdiction: stores unique jurisdictions (regions)
# CrimeType: stores types of crimes (Murder, Rape, etc.)
# CrimeStat: stores stats per jurisdiction, type, year


# Melt the data to long format for insertion (focusing on raw crime counts)
long_crime = violentpropertycrime.melt(
    id_vars=["JURISDICTION", "YEAR"],
    value_vars=["MURDER", "RAPE", "ROBBERY", "AGG. ASSAULT", "B & E", "LARCENY THEFT", "M/V THEFT"],
    var_name="crime_type",
    value_name="count"
)

# Clean crime_type names (title case, replace underscores if any)
long_crime["crime_type"] = long_crime["crime_type"].str.replace("_", " ").str.title()

# Clean population data (remove commas and convert to int)
# Handle both string and numeric population values
if violentpropertycrime["POPULATION"].dtype == 'object':
    violentpropertycrime["POPULATION"] = violentpropertycrime["POPULATION"].str.replace(",", "").astype(int)
else:
    violentpropertycrime["POPULATION"] = violentpropertycrime["POPULATION"].astype(int)

# Ensure there are at least 10 rows per table by checking the data
print(f"Number of jurisdictions: {long_crime['JURISDICTION'].nunique()}")
print(f"Number of crime types: {long_crime['crime_type'].nunique()}")
print(f"Total stats rows: {len(long_crime)}")

# Insert jurisdictions
for jur_name in long_crime["JURISDICTION"].unique():
    Jurisdiction.get_or_create(name=jur_name)

# Insert crime types
for crime_name in long_crime["crime_type"].unique():
    CrimeType.get_or_create(name=crime_name)

# Insert populations
for _, row in violentpropertycrime[["JURISDICTION", "YEAR", "POPULATION"]].drop_duplicates().iterrows():
    jurisdiction = Jurisdiction.get(Jurisdiction.name == row["JURISDICTION"])
    Population.get_or_create(
        jurisdiction=jurisdiction,
        year=row["YEAR"],
        defaults={"population": row["POPULATION"]}
    )

# Insert stats
for _, row in long_crime.iterrows():
    jurisdiction = Jurisdiction.get(Jurisdiction.name == row["JURISDICTION"])
    crime_type = CrimeType.get(CrimeType.name == row["crime_type"])
    
    CrimeStat.get_or_create(
        jurisdiction=jurisdiction,
        crime_type=crime_type,
        year=row["YEAR"],
        defaults={"count": row["count"]}
    )

# Close the database
db.close()

# To verify, query and print some data
db.connect()
print("Crime Stats:")
for stat in CrimeStat.select().limit(5):  # Limit to 5 for brevity
    print(stat.jurisdiction.name, stat.crime_type.name, stat.year, stat.count)
print("\nPopulations:")
for pop in Population.select().limit(5):  # Limit to 5 for brevity
    print(pop.jurisdiction.name, pop.year, pop.population)
db.close()

ValueError: invalid literal for int() with base 10: '79,655'